# Cell-Akamai companion 07 · Concurrency and saturation (client-side)
**Local Ollama on this host — NOT an Akamai GPU run, NOT vLLM.** Original workshop notebook `07_inference_engine_saturation/07_inference_engine_saturation.ipynb` is unchanged; this notebook consumes the same *question* on a different backend. Results are never pooled with Akamai runs. Status: **NOT EXECUTED** until the executed copy exists.

Equal declared request counts per concurrency cell, warmup separate, no overlapping tests, 2 repeats (both reported — no averaged variance). Token counts are Ollama's `eval_count`, **never stream chunks** (the workshop's loadtest counts chunks as a throughput proxy; this notebook does not). Server queue depth / KV pressure are `NOT_AVAILABLE` on Ollama. The server's parallel-slot setting is `UNKNOWN`; the curve shows effective behavior.

In [ ]:
import sys, json, pathlib
sys.path[:0] = ["lane_b/companion", "lane_b/harness"]   # run from the repo root
import cakm, pandas as pd
OUT = pathlib.Path("runs/companion"); OUT.mkdir(parents=True, exist_ok=True)
MODELS = ["hf.co/LiquidAI/LFM2.5-350M-GGUF:Q4_K_M", "hf.co/LiquidAI/LFM2.5-1.2B-Instruct-GGUF:Q4_K_M"]  # small models; edit to taste
HOST = cakm.host_info(); print(json.dumps(HOST, indent=1))

In [ ]:
PROMPT = "List four factors that limit LLM decode throughput, one sentence each. "
CONC = [1, 2, 4, 8]; N = 16; REPEATS = 2; NP = 48; OPT = {"num_ctx": 2048}
cells, raw = [], []
for m in MODELS:
    cakm.generate_stream(m, PROMPT, num_predict=8, options=OPT)   # warmup, excluded
    for c in CONC:
        for rep in range(REPEATS):
            res, wall = cakm.run_concurrent(lambda i: cakm.generate_stream(m, PROMPT + f"(request {i}) ", num_predict=NP, options=OPT), N, c)
            ok = [r for r in res if r["status"] == "OBSERVED"]
            toks = sum(r["eval_count"] for r in ok if r["eval_count"] is not None)
            cells.append({"model": m, "concurrency": c, "repeat": rep, "requests": N, "successful": len(ok), "failed": N - len(ok),
                          "completion_tokens": toks, "batch_wall_s": wall, "aggregate_tok_s": toks / wall if wall else None,
                          "latency_p50_s": cakm.percentile([r["wall_s"] for r in ok], 50), "latency_p95_s": cakm.percentile([r["wall_s"] for r in ok], 95),
                          "ttfc_p50_s": cakm.percentile([r["ttfc_s"] for r in ok], 50), "ttfc_p95_s": cakm.percentile([r["ttfc_s"] for r in ok], 95),
                          "kv_cache_peak": None, "queue_peak": None, "server_ttft_p95": None})
            raw += [{**r, "model": m, "concurrency": c, "repeat": rep} for r in res]
pd.DataFrame(cells)

In [ ]:
rec = cakm.write_receipt(OUT/"07", "saturation_cells", cells, {"module": "07", "host": HOST, "models": MODELS, "requests_per_cell": N,
        "not_available": ["kv_cache_peak", "queue_peak", "server_ttft_p95"], "ollama_parallel_slots": "UNKNOWN"})
cakm.write_receipt(OUT/"07", "saturation_raw_requests", raw, {"module": "07"})
print(rec["jsonl_sha256"])
df = pd.DataFrame(cells); df.pivot_table(index=["model", "concurrency"], columns="repeat", values="aggregate_tok_s")

**Reading it:** if aggregate tokens/s stops rising with concurrency while latency p95 climbs, this *client-side* curve shows saturation of this Ollama server on this Mac. Null columns are Akamai/vLLM-only measurements and must not be filled in from here.